# 3 · Pedotransfer functions (Saxton–Rawls)

When `flags.compute_ptf` is enabled, the soil stage derives hydraulic
parameters from texture and organic matter using the
 [`saxton_rawls`](../api/handlers.md#data4simplace.soil.ptf.saxton_rawls)
 (2006) equations. This notebook runs them on synthetic soils — no
 external data required.

## A single soil

In [ ]:
import xarray as xr
from data4simplace.soil.ptf import saxton_rawls

# sand / clay as percent (0-100); organic matter as percent
sand = xr.DataArray(40.0)
clay = xr.DataArray(20.0)
om   = xr.DataArray(2.5)

hydraulic = saxton_rawls(sand, clay, om)
{name: float(hydraulic[name]) for name in hydraulic.data_vars}

Outputs (see the API docs for units):

- `theta_wp` — wilting point [m³ m⁻³]
- `theta_fc` — field capacity [m³ m⁻³]
- `theta_sat` — saturation [m³ m⁻³]
- `theta_paw` — plant-available water = fc − wp [m³ m⁻³]
- `ksat` — saturated hydraulic conductivity [mm h⁻¹]

## Across the USDA texture range

In [ ]:
import numpy as np

sand_ax = np.linspace(5, 90, 60)
clay_ax = np.linspace(5, 60, 60)
sand2d, clay2d = np.meshgrid(sand_ax, clay_ax)
sand_da = xr.DataArray(sand2d, dims=('clay', 'sand'),
                       coords={'clay': clay_ax, 'sand': sand_ax})
clay_da = xr.DataArray(clay2d, dims=('clay', 'sand'),
                       coords={'clay': clay_ax, 'sand': sand_ax})
# keep only physically valid soils (sand + clay <= 100)
valid = (sand_da + clay_da) <= 100
ptf = saxton_rawls(sand_da.where(valid), clay_da.where(valid))
ptf

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, var, title in zip(
    axes,
    ['theta_wp', 'theta_fc', 'theta_paw'],
    ['Wilting point', 'Field capacity', 'Plant-available water'],
):
    im = ptf[var].plot(ax=ax, cmap='YlGnBu', add_colorbar=True)
    ax.set_title(title)
    ax.set_xlabel('sand %'); ax.set_ylabel('clay %')
plt.tight_layout()

Field capacity and plant-available water rise with clay content, while
sandy soils drain freely — the expected physical behaviour, and a quick
sanity check that the PTF is wired up correctly.

## How the pipeline calls this

In [`Pipeline.run`](../api/pipeline.md#data4simplace.pipeline.Pipeline),
organic matter is derived from SoilGrids organic carbon
(`om ≈ 0.1 × 1.724 × SOC`) before the PTF is applied to the gridded
`sand` and `clay` fields.